# Text preprocessing and feature extraction

This notebook examines the English-language review text **after** it has been cleaned by `src/preprocessing.py`, then shows how that text is converted into numerical features.

The same **group-aware train / validation / test partition** and vectorizer configurations used in `src/train.py` are reused here. Feature vocabularies are fitted on training reviews only. This notebook does not retrain the final classifiers or modify their saved model files.

**Inputs:** `data/processed/energy_app_reviews_cleaned.csv` and `src/train.py`  
**Outputs (existing report filenames):** `reports/feature_extraction_summary.csv`, `reports/top_countvectorizer_features.csv`, `reports/top_tfidf_features.csv`

## 1. Load the prepared reviews

The data collection covers Google Play and Apple App Store reviews from energy-related apps. `src/preprocessing.py` handles English-language filtering, conflicting-label review exclusion, and text cleaning.

Star ratings are mapped to sentiment classes for this project: **1–2 → Negative**, **3 → Neutral**, **4–5 → Positive**. These are *rating-derived proxy labels*, not independently annotated sentiment labels.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.base import clone

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# Works when the notebook is opened from the project's root or notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "train.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "train.py").is_file():
    raise FileNotFoundError("Open this notebook from the project root or notebooks/ directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import make_partitions, sentiment_candidates, rating_candidates

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "energy_app_reviews_cleaned.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

Project: C:\Users\HP\Desktop\nlp-yelp-sentiment-rating-prediction
Dataset: C:\Users\HP\Desktop\nlp-yelp-sentiment-rating-prediction\data\processed\energy_app_reviews_cleaned.csv


In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
required = ["clean_review_text", "sentiment", "rating"]
missing = set(required).difference(df.columns)
if missing:
    raise ValueError(f"Missing expected columns: {sorted(missing)}")

# The same load-time normalization performed by src/train.py.
df = df.dropna(subset=required).copy()
df["clean_review_text"] = (
    df["clean_review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
)
df = df.loc[df["clean_review_text"].ne("")].reset_index(drop=True)
df["rating"] = df["rating"].astype(int)

expected_sentiment = df["rating"].map({
    1: "Negative", 2: "Negative", 3: "Neutral", 4: "Positive", 5: "Positive"
})
assert (df["sentiment"] == expected_sentiment).all(), "Rating / sentiment mapping mismatch"
assert df["rating"].isin([1, 2, 3, 4, 5]).all()

print(f"Processed reviews: {len(df):,}")
print("Applications:", df["app_name"].nunique() if "app_name" in df else "not recorded")
print("Platforms:", df["platform"].value_counts().to_dict() if "platform" in df else "not recorded")
print("Sentiment counts:", df["sentiment"].value_counts().to_dict())
print("Rating counts:", df["rating"].value_counts().sort_index().to_dict())
df[["app_name", "rating", "sentiment", "clean_review_text"]].head(5)

Processed reviews: 70,994
Applications: 19
Platforms: {'google_play': 60766, 'apple_app_store': 10228}
Sentiment counts: {'Negative': 36334, 'Positive': 29121, 'Neutral': 5539}
Rating counts: {1: 29951, 2: 6383, 3: 5539, 4: 6354, 5: 22767}


,app_name,rating,sentiment,clean_review_text
0,Octopus Energy,4,Positive,love app time ever since change accessing coffee voucher frustrating not able get one easily wee...
1,Octopus Energy,3,Neutral,app missing important feature see electricity usage realtime customer smart meter feature availa...
2,Octopus Energy,5,Positive,best electricity provider recommend everyone
3,Octopus Energy,5,Positive,easy mus e sein
4,Octopus Energy,4,Positive,hallo octopus app not provide transaction history n account balance new version please check thank


## 2. Reproduce the training partitions

Rather than creating a fresh random split in the notebook, we call `make_partitions()` directly from `src/train.py`.

Its `StratifiedGroupKFold` partitions are based on the rating labels and **normalized cleaned review text**. Identical cleaned text cannot occur in more than one of the training, validation, and test partitions. This is an exact-text safeguard, not a complete near-duplicate detection system.

The test set is for final evaluation in the training script. Here it is only transformed, **not used to fit the feature vocabulary or choose models**.

In [3]:
groups = df["clean_review_text"].str.casefold().to_numpy()
train_idx, validation_idx, test_idx = make_partitions(df, groups)

text = df["clean_review_text"].to_numpy()
X_train = text[train_idx]
X_validation = text[validation_idx]
X_test = text[test_idx]

assert len(X_train) + len(X_validation) + len(X_test) == len(df)
sets = [set(groups[idx]) for idx in (train_idx, validation_idx, test_idx)]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])

print("Training:", len(X_train))
print("Validation:", len(X_validation))
print("Test:", len(X_test))
print("Cross-partition cleaned-text overlap: 0")

Train      : 42,596 reviews | ratings {1: 17970, 2: 3830, 3: 3324, 4: 3812, 5: 13660}
Validation : 14,199 reviews | ratings {1: 5990, 2: 1277, 3: 1107, 4: 1271, 5: 4554}
Test       : 14,199 reviews | ratings {1: 5991, 2: 1276, 3: 1108, 4: 1271, 5: 4553}
Training: 42596
Validation: 14199
Test: 14199
Cross-partition cleaned-text overlap: 0


## 3. Reuse the final models' feature extractors

The model comparison in `src/train.py` selected these feature types:

| Task | Final classifier | Feature representation |
|---|---|---|
| Sentiment classification | LinearSVC | Word-level TF-IDF (unigrams and bigrams) |
| Rating prediction | Multinomial Naive Bayes | CountVectorizer (unigrams and bigrams) |

The vectorizers below are **cloned from the candidate pipelines** in `src/train.py`, so changes to their settings can be reflected here without copying or guessing parameters.

No extra custom stopword list is applied here: the final pipelines do not use one. This also avoids accidentally discarding meaningful negation words such as *not*.

In [4]:
rating_pipeline = rating_candidates()["CountVectorizer + Naive Bayes"]
sentiment_pipeline = sentiment_candidates()["TF-IDF + LinearSVC"]

count_vectorizer = clone(rating_pipeline.named_steps["vectorizer"])
tfidf_vectorizer = clone(sentiment_pipeline.named_steps["vectorizer"])

print("Rating feature extractor:", count_vectorizer)
print("Sentiment feature extractor:", tfidf_vectorizer)

Rating feature extractor: CountVectorizer(max_df=0.9, max_features=5000, min_df=3, ngram_range=(1, 2))
Sentiment feature extractor: TfidfVectorizer(max_df=0.95, max_features=10000, min_df=2, ngram_range=(1, 2))


In [5]:
# Fit vocabulary and IDF values on the training portion only.
X_train_count = count_vectorizer.fit_transform(X_train)
X_validation_count = count_vectorizer.transform(X_validation)
X_test_count = count_vectorizer.transform(X_test)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_validation_tfidf = tfidf_vectorizer.transform(X_validation)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("CountVectorizer: train / validation / test")
print(X_train_count.shape, X_validation_count.shape, X_test_count.shape)
print("TF-IDF: train / validation / test")
print(X_train_tfidf.shape, X_validation_tfidf.shape, X_test_tfidf.shape)
print("Sparse matrices are kept sparse to avoid unnecessarily using memory.")

CountVectorizer: train / validation / test
(42596, 5000) (14199, 5000) (14199, 5000)
TF-IDF: train / validation / test
(42596, 10000) (14199, 10000) (14199, 10000)
Sparse matrices are kept sparse to avoid unnecessarily using memory.


## 4. Save the feature extraction summary

These dimensions describe the **training-only vocabulary** and its transformation of the other partitions. They are not model-performance metrics. After model selection, `src/train.py` refits the chosen final pipeline on **training + validation**, so that saved model may learn a somewhat different final vocabulary.

In [6]:
def describe_vectorizer(label, task, vec, training_matrix):
    p = vec.get_params()
    return {
        "feature_method": label,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "number_of_features": training_matrix.shape[1],
        "ngram_range": f"{p['ngram_range'][0]} to {p['ngram_range'][1]}",
        "max_features": p["max_features"],
        "min_df": p["min_df"],
        "max_df": p["max_df"],
        "validation_rows": len(validation_idx),
        "task": task,
        "analyzer": p["analyzer"],
    }

feature_summary = pd.DataFrame([
    describe_vectorizer("CountVectorizer", "rating", count_vectorizer, X_train_count),
    describe_vectorizer("TF-IDF", "sentiment", tfidf_vectorizer, X_train_tfidf),
])
feature_summary

,feature_method,train_rows,test_rows,number_of_features,ngram_range,max_features,min_df,max_df,validation_rows,task,analyzer
0,CountVectorizer,42596,14199,5000,1 to 2,5000,3,0.90,14199,rating,word
1,TF-IDF,42596,14199,10000,1 to 2,10000,2,0.95,14199,sentiment,word


In [7]:
summary_path = REPORTS_DIR / "feature_extraction_summary.csv"
feature_summary.to_csv(summary_path, index=False)
print("Updated:", summary_path)

Updated: C:\Users\HP\Desktop\nlp-yelp-sentiment-rating-prediction\reports\feature_extraction_summary.csv


## 5. Inspect common text features

For CountVectorizer, `score` is the **total token/phrase count in the training split**. For TF-IDF, `score` is the **sum of TF-IDF weights over the training split**. These scores are useful for exploring vocabulary, but they are **not classifier coefficients or proof of sentiment importance**.

In [8]:
def top_features(vectorizer, feature_matrix, top_n=25):
    names = vectorizer.get_feature_names_out()
    scores = np.asarray(feature_matrix.sum(axis=0)).ravel()
    return (
        pd.DataFrame({"feature": names, "score": scores})
        .sort_values(["score", "feature"], ascending=[False, True])
        .head(top_n)
        .reset_index(drop=True)
    )

top_count_features = top_features(count_vectorizer, X_train_count)
top_tfidf_features = top_features(tfidf_vectorizer, X_train_tfidf)
print("Top CountVectorizer features (training reviews)")
display(top_count_features.head(15))
print("Top TF-IDF features (training reviews)")
display(top_tfidf_features.head(15))

Top CountVectorizer features (training reviews)


,feature,score
0,app,28765
1,not,26765
2,use,7868
3,work,6935
4,time,6497
5,no,6250
6,easy,5394
7,get,5000
8,meter,4926
9,charging,4649


Top TF-IDF features (training reviews)


,feature,score
0,app,1775.046887
1,not,1679.778852
2,use,1014.380979
3,easy,1010.531017
4,work,778.416424
5,easy use,685.972698
6,time,645.660619
7,great,634.825209
8,good,630.331742
9,no,610.348373


In [9]:
count_path = REPORTS_DIR / "top_countvectorizer_features.csv"
tfidf_path = REPORTS_DIR / "top_tfidf_features.csv"
top_count_features.to_csv(count_path, index=False)
top_tfidf_features.to_csv(tfidf_path, index=False)

print("Updated:", count_path)
print("Updated:", tfidf_path)
print("Saved all three existing feature reports. No models or evaluation reports were overwritten.")

Updated: C:\Users\HP\Desktop\nlp-yelp-sentiment-rating-prediction\reports\top_countvectorizer_features.csv
Updated: C:\Users\HP\Desktop\nlp-yelp-sentiment-rating-prediction\reports\top_tfidf_features.csv
Saved all three existing feature reports. No models or evaluation reports were overwritten.


## 6. Interpretation and limitations

- Count features capture the frequency of terms and phrases; Multinomial Naive Bayes uses them for the selected rating model.
- Word-level TF-IDF reduces the influence of terms that occur throughout the corpus; LinearSVC uses this representation for the selected sentiment model.
- Unigrams and bigrams preserve local expressions such as *not working*, *charging station*, and *customer service*.
- The top-feature tables summarize occurrence/weight, **not** which words cause positive or negative predictions.
- Star-rating-derived sentiment labels may disagree with the nuanced tone of an individual review. Neutral and middle-rating classes remain underrepresented.
- The three CSV files above describe feature extraction. Final model selection, refitting, and held-out test evaluation remain exclusively in `src/train.py` (and are documented in Notebook 03).